# Convergent Probe CBED — Au [110]

Accuracy of each propagation method as a function of probe convergence semi-angle.

- **Ground truth**: Second-Order Wave ODE with 128×128×128 finely sampled potential
- **Test methods**: Fresnel (Paraxial), Angular Spectrum (Non-Paraxial), WPM (fixed 32 slices/cell)
- **Probe angles**: 5, 10, 20, 30, 50, 60, 70, 80, 90, 100, 150, 200, 300 mrad
- **Crystal**: Au [110], 200 keV, ~100 unit cells thick
- **Metric**: Relative L2 norm of diffraction pattern vs Second-Order Wave ODE ground truth


In [ ]:
%matplotlib inline

import os
from time import perf_counter

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".4"

import abtem
import cupy
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from ase.build import bulk, surface
from matplotlib.colors import LogNorm

from wide_angle_propagation.propagation_methods import (
    energy2wavelength,
    fresnel_propagation_kernel,
    angular_spectrum_propagation_kernel,
    simulate_fresnel_as,
    simulate_wpm,
    simulate_kg_ode_full,
)

abtem.config.set({"device": "gpu"})
abtem.config.set({"precision": "float64"})
jax.config.update("jax_enable_x64", True)

## Build crystal, potentials, and helpers

In [ ]:
# --- Parameters ---
energy = 200e3
a_Au = 4.076

# --- Build Au [110] orthogonal cell ---
au_bulk = bulk("Au", "fcc", a=a_Au)
au_110 = surface(au_bulk, (1, 1, 0), layers=2)
unit_cell = abtem.orthogonalize_cell(au_110)

z_period = a_Au / np.sqrt(2)
unit_cell.cell[2, 2] = z_period
unit_cell.pbc = [True, True, True]

nx_rep = int(np.ceil(10 / unit_cell.cell.lengths()[0]))
ny_rep = int(np.ceil(10 / unit_cell.cell.lengths()[1]))
nz_rep = int(np.ceil(100 / z_period))

supercell = unit_cell * (nx_rep, ny_rep, nz_rep)
total_thickness = float(supercell.cell[2, 2])
wavelength = float(energy2wavelength(energy))

# --- ODE reference grid: 128×128 lateral, 128 slices/cell ---
ODE_GPTS = (128, 128)
ODE_SLICES_PER_CELL = 128

semi_angle_mrad = np.array([5, 10, 30, 50, 60, 70, 80, 90, 100, 150, 200, 300], dtype=float)

# --- Z-sampling ---
TEST_SLICES_PER_CELL = 32

dz_ode = z_period / ODE_SLICES_PER_CELL
dz_test = z_period / TEST_SLICES_PER_CELL

print(f"Au [110], {energy/1e3:.0f} keV, λ = {wavelength:.4f} Å")
print(f"Unit cell z-period = {z_period:.4f} Å")
print(f"Supercell: {nx_rep}×{ny_rep}×{nz_rep}, total thickness = {total_thickness:.1f} Å")
print(f"\nConvergence semi-angles: {semi_angle_mrad} mrad")
print(f"ODE ground truth: gpts={ODE_GPTS}, dz = {dz_ode:.4f} Å ({ODE_SLICES_PER_CELL} slices/cell)")
print(f"Test methods:     dz = {dz_test:.4f} Å ({TEST_SLICES_PER_CELL} slices/cell)")

## Build both potentials and define helper functions

In [ ]:
print("Building ODE ground-truth potential (128×128 lateral, 128 slices/cell)...")
pot_ode_abtem = abtem.Potential(
    supercell, gpts=ODE_GPTS, slice_thickness=dz_ode,
    projection="finite", parametrization="lobato",
)
gpts = tuple(pot_ode_abtem.gpts)
sampling = (float(pot_ode_abtem.sampling[0]), float(pot_ode_abtem.sampling[1]))
pot_ode_arr = jnp.array(cupy.asnumpy(pot_ode_abtem.build(lazy=False).array) / dz_ode)
del pot_ode_abtem
cupy.get_default_memory_pool().free_all_blocks()
print(f"  ODE potential shape: {pot_ode_arr.shape}")

print("Building test-method potential (same lateral grid, 32 slices/cell)...")
pot_test_abtem = abtem.Potential(
    supercell, gpts=gpts, slice_thickness=dz_test,
    projection="finite", parametrization="lobato",
)
pot_test_arr = jnp.array(cupy.asnumpy(pot_test_abtem.build(lazy=False).array) / dz_test)
del pot_test_abtem
cupy.get_default_memory_pool().free_all_blocks()
print(f"  Test potential shape: {pot_test_arr.shape}")

fk = jnp.array(fresnel_propagation_kernel(*gpts, sampling, z=dz_test, energy=energy))
ak = jnp.array(angular_spectrum_propagation_kernel(*gpts, sampling, z=dz_test, energy=energy))

ny_g, nx_g = gpts
print(f"Grid: {gpts}, sampling = ({sampling[0]:.4f}, {sampling[1]:.4f}) Å")

def make_convergent_probe(semi_angle_mrad_val, ny, nx, samp_y, samp_x, wavelength_ang):
    alpha = semi_angle_mrad_val * 1e-3
    k_max = np.sin(alpha) / wavelength_ang
    fy = np.fft.fftfreq(ny, d=samp_y)
    fx = np.fft.fftfreq(nx, d=samp_x)
    Fx, Fy = np.meshgrid(fx, fy)
    aperture = (np.sqrt(Fx**2 + Fy**2) <= k_max).astype(np.complex128)
    n_pix = float(aperture.real.sum())
    if n_pix > 0:
        aperture /= np.sqrt(n_pix)
    probe = np.fft.fftshift(np.fft.ifft2(aperture))
    return jnp.array(probe, dtype=jnp.complex128)

print("Helper functions defined.")


## Sweep convergence semi-angle

For each angle: build probe, run KG ODE ground truth (fine 128³ potential), run 4 test methods (coarse slicing), compute relative L2 error vs ODE.

In [ ]:
method_names = ["Fresnel MS", "Angular Spectrum", "WPM"]

rel_l2_results = {name: [] for name in method_names}
timings = {name: [] for name in method_names}
gt_timings = []

for alpha in semi_angle_mrad:
    print(f"
{'='*60}")
    print(f"Convergence semi-angle = {alpha:.0f} mrad")
    print(f"{'='*60}")

    probe = make_convergent_probe(alpha, ny_g, nx_g, sampling[0], sampling[1], wavelength)

    # Ground truth: KG ODE (full second-order) with fine 128³ potential
    print(f"  KG ODE ground truth (dz={dz_ode:.4f} Å, {ODE_SLICES_PER_CELL} sl/cell)...", end=" ", flush=True)
    t0 = perf_counter()
    ew_gt, _, dp_gt_raw, _ = simulate_kg_ode_full(
        pot_ode_arr, probe, dz_ode, energy, sampling,
        save_wavefronts=False,
    )
    dp_gt = np.asarray(dp_gt_raw)
    t_gt = perf_counter() - t0
    gt_timings.append(t_gt)
    print(f"{t_gt:.1f}s")

    # Test methods (coarser slicing)
    methods_to_run = {
        "Fresnel MS":       lambda p=probe: simulate_fresnel_as(pot_test_arr, p, fk, dz_test, energy),
        "Angular Spectrum": lambda p=probe: simulate_fresnel_as(pot_test_arr, p, ak, dz_test, energy),
        "WPM":              lambda p=probe: simulate_wpm(pot_test_arr, p, dz_test, energy, sampling),
    }

    for name in method_names:
        print(f"  {name}...", end=" ", flush=True)
        t0 = perf_counter()
        result = methods_to_run[name]()
        elapsed = perf_counter() - t0

        ew = np.asarray(result[0])
        dp = np.abs(np.fft.fftshift(np.fft.fft2(ew))) ** 2
        dp_diff = dp - dp_gt
        rel_l2 = float(np.linalg.norm(dp_diff) / np.linalg.norm(dp_gt))

        rel_l2_results[name].append(rel_l2)
        timings[name].append(elapsed)

        print(f"{elapsed:.1f}s, Rel L2 = {rel_l2:.4e}")

print(f"
{'='*60}")
print("Sweep complete.")


## Accuracy vs convergence semi-angle

In [ ]:
display_names = {
    "Fresnel MS": "Fresnel (Paraxial)",
    "Angular Spectrum": "Angular Spectrum (Non-Paraxial)",
    "WPM": "WPM (Wide-angle, local n)",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

markers = {'Fresnel MS': 'o', 'Angular Spectrum': 's', 'WPM': 'D'}
colors = {'Fresnel MS': 'C0', 'Angular Spectrum': 'C1', 'WPM': 'C2'}

# Left: Rel L2 vs semi-angle
for name in method_names:
    ax1.semilogy(
        semi_angle_mrad,
        rel_l2_results[name],
        f'-{markers[name]}',
        color=colors[name],
        label=display_names.get(name, name),
        markersize=7,
        linewidth=1.5,
    )

ax1.set_xlabel("Convergence semi-angle (mrad)", fontsize=12)
ax1.set_ylabel("Relative L2 error vs Second-Order Wave ODE", fontsize=12)
ax1.set_title(
    f"DP error vs Second-Order Wave ODE reference
Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å, "
    f"test @ {TEST_SLICES_PER_CELL} sl/cell",
    fontsize=11,
 )
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Right: runtime vs semi-angle
for name in method_names:
    ax2.plot(
        semi_angle_mrad,
        timings[name],
        f'-{markers[name]}',
        color=colors[name],
        label=display_names.get(name, name),
        markersize=7,
        linewidth=1.5,
    )

ax2.set_xlabel("Convergence semi-angle (mrad)", fontsize=12)
ax2.set_ylabel("Runtime (s)", fontsize=12)
ax2.set_title("Compute cost vs convergence semi-angle", fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# Export figure for paper
from pathlib import Path

def resolve_paper_fig_dir():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "Paper" / "figures"
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
    raise FileNotFoundError("Could not locate Paper/figures from the current working directory")

paper_fig_dir = resolve_paper_fig_dir()
output_path = paper_fig_dir / "angle_convergence.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)
if not output_path.exists() or output_path.stat().st_size == 0:
    raise RuntimeError(f"Failed to write figure: {output_path}")
print(f"Saved -> {output_path}")

## Diffraction pattern comparison — selected convergence angles

Rows: 5, 30, 50, and 100 mrad. Columns: ODE ground truth plus test methods.

In [ ]:
display_names = {
    "Fresnel MS": "Fresnel (Paraxial)",
    "Angular Spectrum": "Angular Spectrum (Non-Paraxial)",
    "WPM": "WPM (Wide-angle, local n)",
}

selected_angles_mrad = [5.0, 30.0, 50.0, 100.0]
selected_indices = [int(np.where(semi_angle_mrad == a)[0][0]) for a in selected_angles_mrad]

fig, axes = plt.subplots(
    len(selected_angles_mrad), len(method_names) + 1,
    figsize=(4 * (len(method_names) + 1), 3.5 * len(selected_angles_mrad))
)

for row, (alpha, idx) in enumerate(zip(selected_angles_mrad, selected_indices)):
    probe = make_convergent_probe(alpha, ny_g, nx_g, sampling[0], sampling[1], wavelength)

    ew_gt_here, _, dp_gt_raw_here, _ = simulate_kg_ode_full(
        pot_ode_arr, probe, dz_ode, energy, sampling,
        save_wavefronts=False,
    )
    dp_gt_here = np.asarray(dp_gt_raw_here)

    col_entries = []
    for name in method_names:
        if name == "Fresnel MS":
            result = simulate_fresnel_as(pot_test_arr, probe, fk, dz_test, energy)
        elif name == "Angular Spectrum":
            result = simulate_fresnel_as(pot_test_arr, probe, ak, dz_test, energy)
        elif name == "WPM":
            result = simulate_wpm(pot_test_arr, probe, dz_test, energy, sampling)
        dp = np.abs(np.fft.fftshift(np.fft.fft2(np.asarray(result[0])))) ** 2
        col_entries.append((name, dp))

    vmax = np.percentile(dp_gt_here, 99.9)
    vmin = max(vmax * 1e-5, dp_gt_here[dp_gt_here > 0].min())

    axes[row, 0].imshow(dp_gt_here, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="inferno", origin="lower")
    axes[row, 0].set_title(f"Second-Order Wave ODE
{alpha:.0f} mrad", fontsize=9)
    axes[row, 0].axis("off")

    for col, (name, dp) in enumerate(col_entries):
        rl2 = rel_l2_results[name][idx]
        display_name = display_names.get(name, name)
        axes[row, col + 1].imshow(dp, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="inferno", origin="lower")
        axes[row, col + 1].set_title(f"{display_name}
Rel L2={rl2:.2e}", fontsize=9)
        axes[row, col + 1].axis("off")

plt.suptitle(
    f"CBED patterns — Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å
"
    f"Second-Order Wave ODE reference ({ODE_SLICES_PER_CELL} sl/cell), test methods at {TEST_SLICES_PER_CELL} sl/cell",
    fontsize=11, y=1.01,
 )
plt.tight_layout()
plt.show()


In [ ]:
# Export CBED pattern grid for paper
from pathlib import Path

def resolve_paper_fig_dir():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "Paper" / "figures"
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
    raise FileNotFoundError("Could not locate Paper/figures from the current working directory")

paper_fig_dir = resolve_paper_fig_dir()
output_path = paper_fig_dir / "cbed_pattern_grid.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)
if not output_path.exists() or output_path.stat().st_size == 0:
    raise RuntimeError(f"Failed to write figure: {output_path}")
print(f"Saved -> {output_path}")

## Summary table

In [ ]:
print(f"Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å")
print(f"Ground truth: KG ODE (full 2nd-order) at gpts={ODE_GPTS}, dz = {dz_ode:.4f} Å ({ODE_SLICES_PER_CELL} sl/cell)")
print(f"Test methods: dz = {dz_test:.4f} Å ({TEST_SLICES_PER_CELL} slices/cell)\n")

hdr = f"{'Semi-angle (mrad)':>18s}"
for name in method_names:
    hdr += f" {name:>17s}"
hdr += f" {'ODE GT time (s)':>16s}"
print(hdr)
print("-" * len(hdr))

for i, alpha in enumerate(semi_angle_mrad):
    row = f"{alpha:>18.0f}"
    for name in method_names:
        row += f" {rel_l2_results[name][i]:>17.4e}"
    row += f" {gt_timings[i]:>16.1f}"
    print(row)